In [14]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint

In [15]:
tf.random.set_seed(42)
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [16]:
print("GPUs:", tf.config.list_physical_devices("GPU"))

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [17]:
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [22]:
inputs = tf.keras.layers.Input(shape=(512, 512, 2))

In [23]:
x = tf.keras.layers.Conv2D(
    3,
    kernel_size=1,
    padding="same",
    name="sar_to_rgb"
)(inputs)

In [24]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(512, 512, 3)
)

In [25]:
x = base_model(x)

In [26]:
base_model.trainable = False

In [27]:
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dense(20, activation="relu")(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

In [28]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [29]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 512, 512, 2)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sar_to_rgb (Conv2D)             │ (None, 512, 512, 3)    │             9 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 16, 16, 1280)   │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         2,580 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,221,269 (16.10 MB)

 Trainable params: 169,138 (660.70 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [30]:
def load_sar_tiff(path):
    def _read(p):
        with rasterio.open(p.decode()) as src:
            vv = src.read(1).astype(np.float32)
            vh = src.read(2).astype(np.float32)

        vv = np.clip(vv, -35, 5)
        vh = np.clip(vh, -40, 0)

        vv = (vv + 35) / 40
        vh = (vh + 40) / 40

        img = np.stack([vv, vh], axis=-1)
        img = tf.image.resize(img, IMG_SIZE).numpy()
        return img

    img = tf.numpy_function(_read, [path], tf.float32)
    img.set_shape([512, 512, 2])
    return img

In [32]:
def augment(image, label):
    # flips
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    print("flip done")
    # rotation
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    print("rotation done")
    # translation (pure TensorFlow, no variables)
    tx = tf.random.uniform([], -0.05, 0.05)
    ty = tf.random.uniform([], -0.05, 0.05)
    print("translation done")
    image = tf.roll(
        image,
        shift=[
            tf.cast(tx * IMG_SIZE[0], tf.int32),
            tf.cast(ty * IMG_SIZE[1], tf.int32)
        ],
        axis=[0, 1]
    )

    return image, label

In [35]:
import os
import random
import numpy as np
import tensorflow as tf
import rasterio
import wandb


from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [36]:
IMG_SIZE = (512, 512)
BATCH_SIZE = 8
EPOCHS = 5
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# BASE_PATH = "/Volumes/Windows8_OS/Dataset/Dataset-OG"
BASE_PATH = "/Volumes/Windows8_OS/Dataset/Dataset-OG"

TRAIN_IMG_DIR = os.path.join(BASE_PATH, "Train", "Images")
TEST_IMG_DIR  = os.path.join(BASE_PATH, "Test", "Images")  # if exist

In [39]:
def build_balanced_dataset(images_root, target_per_class=1200):
    oil_dir = os.path.join(images_root, "Oil")
    no_oil_dir = os.path.join(images_root, "No_Oil")
    lookalike_dir = os.path.join(images_root, "Lookalike")

    oil_files = [os.path.join(oil_dir, f) for f in os.listdir(oil_dir) if f.endswith(".tif")]
    no_oil_files = [os.path.join(no_oil_dir, f) for f in os.listdir(no_oil_dir) if f.endswith(".tif")]
    lookalike_files = [os.path.join(lookalike_dir, f) for f in os.listdir(lookalike_dir) if f.endswith(".tif")]

    # Merge No_Oil + Lookalike
    combined_no_oil = no_oil_files + lookalike_files
    random.shuffle(combined_no_oil)

    oil_files = oil_files[:target_per_class]
    combined_no_oil = combined_no_oil[:target_per_class]

    paths = oil_files + combined_no_oil
    labels = [1]*len(oil_files) + [0]*len(combined_no_oil)

    return np.array(paths), np.array(labels)

In [40]:
def build_test_dataset(images_root):
    oil_dir = os.path.join(images_root, "Oil")
    no_oil_dir = os.path.join(images_root, "No_Oil")
    lookalike_dir = os.path.join(images_root, "Lookalike")

    oil_files = [os.path.join(oil_dir, f) for f in os.listdir(oil_dir) if f.endswith(".tif")]
    no_oil_files = [os.path.join(no_oil_dir, f) for f in os.listdir(no_oil_dir) if f.endswith(".tif")]
    lookalike_files = [os.path.join(lookalike_dir, f) for f in os.listdir(lookalike_dir) if f.endswith(".tif")]

    paths = oil_files + no_oil_files + lookalike_files
    labels = (
        [1] * len(oil_files) +
        [0] * len(no_oil_files) +
        [0] * len(lookalike_files)
    )

    return np.array(paths), np.array(labels)

In [42]:
from sklearn.model_selection import train_test_split

all_paths, all_labels = build_balanced_dataset(TRAIN_IMG_DIR)

print("Total balanced samples:", len(all_paths))

Total balanced samples: 2400


In [44]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths,
    all_labels,
    train_size=0.66,
    stratify=all_labels,
    random_state=SEED
)

print("Train samples:", len(train_paths))
print("Val samples:", len(val_paths))

print("Train Oil:", np.sum(train_labels == 1))
print("Train No_Oil:", np.sum(train_labels == 0))
print("Val Oil:", np.sum(val_labels == 1))
print("Val No_Oil:", np.sum(val_labels == 0))

Train samples: 1584
Val samples: 816
Train Oil: 792
Train No_Oil: 792
Val Oil: 408
Val No_Oil: 408


In [45]:
test_paths, test_labels = build_test_dataset(TEST_IMG_DIR)

print("Test samples:", len(test_paths))  # 450
print("Test Oil:", np.sum(test_labels == 1))        # 150
print("Test No_Oil:", np.sum(test_labels == 0))     # 300

Test samples: 450
Test Oil: 150
Test No_Oil: 300


In [46]:
def make_dataset(paths, labels, augment_fn=None, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(len(paths), seed=42)

    ds = ds.map(
        lambda x, y: (load_sar_tiff(x), y),
        num_parallel_calls=1  # rasterio-safe
    )

    if augment_fn is not None:
        ds = ds.map(augment_fn, num_parallel_calls=1)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(1)

    return ds

In [47]:
train_ds = make_dataset(
    train_paths,
    train_labels,
    augment_fn=augment,
    shuffle=True
)

val_ds = make_dataset(
    val_paths,
    val_labels,
    augment_fn=augment,
    shuffle=False
)

test_ds = make_dataset(
    test_paths,
    test_labels,
    augment_fn=None,
    shuffle=False
)

flip done
rotation done
translation done
flip done
rotation done
translation done


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath="EfficientNet-B0_checkpoints/model_epoch_{epoch}.weights.h5",
    save_weights_only=True,
    save_best_only=False
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop, checkpoint_cb]
)

Epoch 1/5
198/198 ━━━━━━━━━━━━━━━━━━━━ 3024s 15s/step - accuracy: 0.4934 - loss: 0.7760 - val_accuracy: 0.5000 - val_loss: 0.6937
Epoch 2/5
